In [7]:
from IPython.display import HTML, Image, display

html = """

<div id="cover-page-container">
    <div id="title-container">
        <h1>Indicators</h1>
        <div id="meta-data-container">
            <div>July 6, 2026</div>
            <div>by Aaron Hardy</div>
        </div>
    </div>
    <div id="title-bar"></div>
</div>
"""

display(HTML(html))

In [3]:
from IPython.display import HTML

HTML(
    """
    <link href="https://fonts.googleapis.com/css2?family=Roboto:wght@400;700&display=swap" rel="stylesheet">

    <link rel="stylesheet" href="custom.css">
    <script src="custom.js" defer></script>
    <style>
        div.jp-OutputPrompt.jp-OutputArea-prompt {
            display: none !important;
        }
    
        .jp-OutputPrompt {
            display: none !important;
        }

        .jp-Notebook-cell.celltag_remove_cell {
            display: none !important;
        }
    </style>

    """
)

# Project Setup

This project involves an end-to-end process of gathering historical stock fundamentals data and generating a stock trading strategy from it using SQL and python. The raw data was sourced from [SimFin.com](https://www.simfin.com/en/fundamental-data-download/), which you will need a free account to download. MySQL was used as the database management system.

## Load Data
The raw data existed as CSVs. Each CSV was loaded into a MySQL table. The code used to load each table can be seen [here](load_data.html)

In [ ]:
USE fundamentals_data;

In [ ]:
SELECT * FROM industries;

In [ ]:
SELECT * FROM us_balance_quarterly

The list of U.S. companies Excel file was inspected in Excel and the file contained rows of data split over several cells in an unpredictable way. In Excel, the CONCAT() function was used to convert all rows to a single concatenated string which was then split again using Data > Text to Columns tool.

Next, we uploaded each SimFin CSV file into a separate SQL database table. The example below shows how the table `us_companies` was uploaded to the MySQL database titled 'fundamentals_data'. All SimFin data files were uploaded similarly.

In [ ]:

DROP TABLE IF EXISTS us_companies;

-- Create the table structure
CREATE TABLE us_companies (
    Ticker VARCHAR(255),
    SimFinId INT,
    `Company Name` VARCHAR(255),
    IndustryId INT,
    ISIN VARCHAR(255),
    `End of financial year (month)` INT,
    `Number Employees` INT,
    `Business Summary` TEXT,
    Market VARCHAR(255),
    CIK INT,
    `Main Currency` VARCHAR(255)
);

-- Upload the csv data
LOAD DATA LOCAL INFILE 'c:\\Users\\aaron\\Desktop\\Local Projects\\Portfolio Projects\\data\\us-companies\\us-companies.csv'
INTO TABLE us_companies
FIELDS TERMINATED BY ','
OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES; -- skip header row


The company profile table (`us_companies`) had 37 rows where both the Ticker and Company name were NULL

In [ ]:
SELECT * FROM us_companies;

UPDATE us_companies
SET `Company Name` = NULL, `Ticker` = NULL
WHERE `Company Name` = '' AND `Ticker` = '';

SELECT COUNT(*) 
FROM us_companies
WHERE (Ticker IS NULL AND `Company Name` IS NULL);


Next, we will delete from the table any rows that have a null value for Ticker and Company Name and therefore insufficient identifying information.

In [ ]:

DELETE FROM us_companies 
WHERE (Ticker IS NULL AND `Company Name` IS NULL);

The data includes several financial statement tables. The dates for a balance sheet table (`us_balance_quarterly`) were observed below.

In [ ]:

SELECT 
    MIN(`Report Date`) as min_date,
    MAX(`Report Date`) as max_date,
    TIMESTAMPDIFF(DAY, MIN(`Report Date`), MAX(`Report Date`)) / 365.25 AS years
FROM us_balance_quarterly;


The data spans a period starting *July 31, 2020* and ending *May 31, 2025*, covering about 4.8 years.

Next, we look at the stock price table (`us_shareprices_monthly`) and 

In [ ]:

-- Clean daily prices
SELECT 
    MIN(Close),
    MAX(Close),
    MIN(`Adj. Close`),
    MAX(`Adj. Close`)
FROM us_shareprices_daily;

The `Close` price ranged from \\$0 to \\$99,999,999.99. 
The `Adj. Close` price ranged from \\$0 to \\$9,984,240.00.

We filtered removed outliers and high-priced stocks to make the analysis accessible to the mainstream investor with budgets of \\$100,000 or less. All tickers whose `Close` price exceeded \\$100,000 on any day over the data period were filtered out.

In [ ]:

DROP TABLE IF EXISTS us_shareprices_daily_filtered

CREATE TABLE us_shareprices_daily_filtered AS 
SELECT * FROM us_shareprices_daily
WHERE Ticker NOT IN (
    SELECT DISTINCT Ticker 
    FROM us_shareprices_daily 
    WHERE `Close` > 100000
)

-- Check that no prices exceed $100,000
SELECT * 
FROM us_shareprices_daily_filtered
WHERE `Close` > 100000

## Metrics

In [ ]:
SELECT
    cfq.Ticker,
    cfq.`Report Date`,
    cfq.`Net Cash from Operating Activities`,
    isq.`Net Income`,
    isq.`Operating Income (Loss)`
    -- isq.`Net Income`/cfq.`Net Cash from Operating Activities` AS CashFlowQuality
FROM
    us_cashflow_quarterly cfq
    LEFT JOIN us_income_quarterly isq ON cfq.Ticker = isq.Ticker
    AND cfq.`Report Date` = isq.`Report Date`
    -- WHERE isq.`Net Income`/cfq.`Net Cash from Operating Activities` IS NOT NULL
UNION

SELECT
    cfq.Ticker,
    cfq.`Report Date`,
    cfq.`Net Cash from Operating Activities`,
    isq.`Net Income`,
    isq.`Operating Income (Loss)`
FROM
    us_cashflow_quarterly cfq
    RIGHT JOIN us_income_quarterly isq ON cfq.Ticker = isq.Ticker
    AND cfq.`Report Date` = isq.`Report Date`
LIMIT 10;

In [ ]:

-- WHERE rn = (
--     SELECT 
--         Ticker,
--         ROW_NUMBER() OVER (PARTITION BY Ticker ORDER BY Date ASC) AS rn 
-- )
-- where `Adj. Close` > 1E6;


-- --------------
-- MONTHLY PRICES
-- --------------

drop table if exists us_shareprices_monthly;
-- When you drop the table, all indexes disappear automatically, so you must recreate them.

CREATE TABLE us_shareprices_monthly AS
SELECT
    Ticker,
    MAX(Date) AS Date,
    -- GROUP_CONCAT() → builds a list of Close prices
	-- ORDER BY Date DESC → newest price first
	-- SUBSTRING_INDEX(..., ',', 1) → pick the first price
    SUBSTRING_INDEX(
        GROUP_CONCAT(Close ORDER BY Date DESC),
        ',',
        1
    ) AS Close,
    SUBSTRING_INDEX(
        GROUP_CONCAT(`Adj. Close` ORDER BY Date DESC),
        ',',
        1
    ) AS AdjClose
FROM us_shareprices_daily
GROUP BY
    Ticker,
    YEAR(Date),
    MONTH(Date);
    
    
CREATE INDEX idx_month_end_ticker_date
ON us_shareprices_monthly (Ticker, Date);
